In [1]:
#install the required modules and library
!pip install -U demucs pydrive2 torchcodec ffmpeg-python



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 8.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.8 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=c6bb02a1819b8b7c505c568def8476861788c57634adcd5d5cc65a2293f8130a
  Stored in directory: /root/.cache/pip/wheels/1b/0c/20/a3b3daa1f9b65c8b0445729f

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import shutil
import subprocess

# Mounted Drive folders
INPUT_FOLDER = '/content/drive/My Drive/BGMRemoverInput'
OUTPUT_FOLDER = '/content/drive/My Drive/SegmentationInput'


In [4]:
# List all audio files (filter .wav, .mp3, .m4a)
file_list = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(('.wav','.mp3','.webm'))]
print(f"🎵 Found {len(file_list)} audio files")


🎵 Found 742 audio files


In [6]:
import torch

# Helper to clean local files generated by Demucs
def clean_workspace(temp_files=[]):
    if os.path.exists("separated"):
        shutil.rmtree("separated")
    for f in temp_files:
        if os.path.exists(f):
            os.remove(f)

# Determine device for Demucs
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Process files one by one
processed = set(os.listdir(OUTPUT_FOLDER))

for idx, input_filename in enumerate(file_list, 1):

    output_vocals = f"{os.path.splitext(input_filename)[0]}_vocals_16k.wav"

    if output_vocals in processed:
        print(f"⏭️ Skipping already processed file: {input_filename}")
        continue

    print(f"\n🔄 Processing file {idx}/{len(file_list)}: {input_filename}")

    temp_files = []

    # Copy input file to working directory
    input_path = os.path.join(INPUT_FOLDER, input_filename)
    shutil.copy(input_path, input_filename)
    temp_files.append(input_filename)

    # Convert any audio file to WAV 16kHz first
    wav_16k = f"{os.path.splitext(input_filename)[0]}_16k.wav"
    subprocess.run([
        "ffmpeg", "-y",
        "-i", input_filename,
        "-ac", "1",
        "-ar", "16000",
        wav_16k
    ], check=True)
    temp_files.append(wav_16k)

    # 🎤 Run Demucs (vocals only) on the converted WAV
    subprocess.run([
        "demucs",
        "--two-stems=vocals",
        "--device", device,
        wav_16k
    ], check=True)

    # 📁 Locate Demucs output
    demucs_base = "separated/htdemucs"
    track_name = os.listdir(demucs_base)[0]
    vocals_path = os.path.join(demucs_base, track_name, "vocals.wav")

    # 🔊 Save final vocals as WAV 16kHz in output folder
    output_vocals = f"{os.path.splitext(input_filename)[0]}_vocals_16k.wav"
    output_path = os.path.join(OUTPUT_FOLDER, output_vocals)
    subprocess.run([
        "ffmpeg", "-y",
        "-i", vocals_path,
        "-ac", "1",
        "-ar", "16000",
        output_path
    ], check=True)

    processed.add(output_vocals)
    print("✅ Saved vocals to:", output_path)

    # 🧹 Clean local working files for this loop
    clean_workspace(temp_files)
    print("🗑️ Local temporary files cleaned")

print("\n🎉 ALL FILES PROCESSED SUCCESSFULLY")

Using device: cuda
⏭️ Skipping already processed file: binancewebm.webm
⏭️ Skipping already processed file: audio-1766566676216.webm
⏭️ Skipping already processed file: audio-1766566744763.webm
⏭️ Skipping already processed file: audio-1766566883023.webm
⏭️ Skipping already processed file: audio-1766566960959.webm
⏭️ Skipping already processed file: audio-1766567069745.webm
⏭️ Skipping already processed file: audio-1766567168350.webm
⏭️ Skipping already processed file: audio-1766567202468.webm
⏭️ Skipping already processed file: audio-1766567259378.webm
⏭️ Skipping already processed file: audio-1766567366978.webm
⏭️ Skipping already processed file: audio-1766567423139.webm
⏭️ Skipping already processed file: audio-1766567503713.webm
⏭️ Skipping already processed file: audio-1766567539227.webm
⏭️ Skipping already processed file: audio-1766567772445.webm
⏭️ Skipping already processed file: audio-1766567859431.webm
⏭️ Skipping already processed file: audio-1766567948144.webm
⏭️ Skipping a